# Logistic Regression Model Implementation

In this notebook, we build a **Logistic Regression classifier from scratch** using only NumPy and Pandas. We apply it to the Bank Marketing dataset to predict whether a customer will subscribe to a term deposit. 

## Problem Formulation

This is a **binary classification** problem. Given customer attributes $\mathbf{x} \in \mathbb{R}^d$, we want to predict whether the customer will subscribe to a term deposit:

$$y \in \{0, 1\}$$

where $y = 1$ means "yes" (subscribed) and $y = 0$ means "no".

Logistic Regression models the conditional probability of class 1 directly:

$$P(y = 1 \mid \mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x} + b)$$

where $\mathbf{w} \in \mathbb{R}^d$ is the weight vector, $b$ is the bias term, and $\sigma$ is the sigmoid activation. The model learns $\mathbf{w}$ and $b$ by minimising the Binary Cross-Entropy loss via gradient descent.

## Sigmoid Function

The **sigmoid function** maps any real number to the interval $(0, 1)$, making its output interpretable as a probability:

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad z = \mathbf{w}^T \mathbf{x} + b$$

**Key properties:**
- $\sigma(z) \to 1$ as $z \to +\infty$ — the model is highly confident the sample belongs to class 1.
- $\sigma(z) \to 0$ as $z \to -\infty$ — the model is highly confident the sample belongs to class 0.
- $\sigma(0) = 0.5$ — the natural midpoint; the default decision boundary.

The output $\hat{p} = \sigma(z)$ is the model's estimated probability that the customer will subscribe.

## Binary Cross-Entropy Loss

The model is trained by minimising the **Binary Cross-Entropy (BCE)** loss, which penalises confident wrong predictions heavily:

$$\mathcal{L} = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]$$

**Interpretation:**
- When $y_i = 1$ and $\hat{p}_i \approx 1$: loss $\approx 0$ (correct, confident prediction).
- When $y_i = 1$ and $\hat{p}_i \approx 0$: loss $\to \infty$ (wrong, confident prediction — heavily penalised).
- The BCE loss is **convex**, so gradient descent converges to the global minimum.

A small numerical constant $\epsilon = 10^{-15}$ is added inside the logarithm to avoid $\log(0)$ instability.

## Gradient Descent, Class Weighting & L2 Regularization

### Gradient Descent
We minimise the BCE loss by iteratively updating the weights in the direction of the negative gradient:

$$\mathbf{w} \leftarrow \mathbf{w} - \alpha \cdot \nabla_{\mathbf{w}} \mathcal{L}$$

where $\alpha = 0.5$ is the **learning rate**. The gradient of the BCE loss is:

$$\nabla_{\mathbf{w}} \mathcal{L} = \frac{1}{n} \mathbf{X}^T (\hat{\mathbf{p}} - \mathbf{y})$$

### Class Weighting
The dataset is heavily imbalanced (~88 % negative, ~12 % positive). Each positive sample's gradient contribution is upweighted by $\frac{n_{\text{neg}}}{n_{\text{pos}}} \approx 7.5$ to prevent the model from ignoring the minority class:

$$\nabla_{\mathbf{w}} \mathcal{L} = \frac{1}{n} \mathbf{X}^T \left[ (\hat{\mathbf{p}} - \mathbf{y}) \odot \mathbf{c} \right], \qquad c_i = \begin{cases} n_{\text{neg}} / n_{\text{pos}} & \text{if } y_i = 1 \\ 1 & \text{otherwise} \end{cases}$$

### L2 Regularization
An L2 penalty is added to prevent overfitting by shrinking weights towards zero:

$$\nabla_{\mathbf{w}} \mathcal{L}_{\text{reg}} = \nabla_{\mathbf{w}} \mathcal{L} + \frac{\lambda}{n} \mathbf{w}$$

The **bias term** (index 0) is excluded from regularisation, as penalising the intercept is generally not beneficial. We use $\lambda = 0.01$.

## Step 1: Load Dataset and Prepare Features

We load `cleaned_data.csv`, produced by `Bank_Marketing_Preprocessing.ipynb`. That notebook:
- Applied one-hot encoding to categorical variables
- Normalized numerical features
- Removed the `duration` column to prevent data leakage

We extract the feature matrix $X$ and binary target $y$, then **prepend a column of ones** to absorb the bias term $b$ into the weight vector, allowing the linear combination $z = \mathbf{w}^T \mathbf{x}$ to naturally include the intercept as $w_0$, with no special-case code.

A **stratified 80/20 train/test split** is applied to preserve the original class ratio (~88 % negative, ~12 % positive) in both sets.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the dataset
df = pd.read_csv('cleaned_data.csv')

# 2. Extract features and target
# 'y_binary' is our target (1 for yes, 0 for no)
X_raw = df.drop(columns=['y', 'y_binary']).copy()
y = df['y_binary'].values.reshape(-1, 1)

# Convert boolean columns to float (0.0/1.0)
for col in X_raw.columns:
    if X_raw[col].dtype == 'bool':
        X_raw[col] = X_raw[col].astype(float)

# 3. Add Bias Term (Intercept)
# We append a column of 1s to X so that the bias 'b' is treated as a weight
X = np.hstack([np.ones((X_raw.shape[0], 1)), X_raw.values])

# 4. Stratified Train-Test Split (80/20)
# Separate indices by class to ensure equal positive/negative ratio in both sets
pos_idx = np.where(y.flatten() == 1)[0]
neg_idx = np.where(y.flatten() == 0)[0]

np.random.seed(42)
np.random.shuffle(pos_idx)
np.random.shuffle(neg_idx)

n_pos_test = int(len(pos_idx) * 0.2)
n_neg_test = int(len(neg_idx) * 0.2)

test_idx  = np.concatenate([pos_idx[:n_pos_test],  neg_idx[:n_neg_test]])
train_idx = np.concatenate([pos_idx[n_pos_test:],  neg_idx[n_neg_test:]])

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size:  {X_test.shape[0]}")
print(f"Train positive rate: {y_train.mean():.3f}")
print(f"Test  positive rate: {y_test.mean():.3f}")

## Step 1b: Post-Split Standardization

Continuous features are **standardized** (zero mean, unit variance) using statistics computed from the **training set only**. Applying training-set means and standard deviations to the test set prevents data leakage.

Only the six continuous features (`age`, `balance`, `day_of_week`, `campaign`, `pdays_clean`, `previous`) are standardised; binary and one-hot encoded columns are already on a 0/1 scale and require no rescaling.

In [ ]:
# 5. Post-split Standardization
# Only continuous columns are scaled; binary/one-hot columns are left as 0/1.
# Mean and std computed from X_train only to avoid data leakage.
continuous_cols = ["age", "balance", "day_of_week", "campaign", "pdays_clean", "previous"]

col_names = list(X_raw.columns)
cont_idx = [col_names.index(c) + 1 for c in continuous_cols if c in col_names]

train_mean = X_train[:, cont_idx].mean(axis=0)
train_std  = X_train[:, cont_idx].std(axis=0) + 1e-8

X_train[:, cont_idx] = (X_train[:, cont_idx] - train_mean) / train_std
X_test[:, cont_idx]  = (X_test[:, cont_idx]  - train_mean) / train_std

print(f"Standardized {len(cont_idx)} continuous features: {continuous_cols}")

## Step 2: Define Model and Train

We implement three core functions from scratch:

1. **`sigmoid(z)`** — applies the sigmoid activation to convert the linear score $z$ into a probability in $(0, 1)$.
2. **`compute_loss(y_true, y_pred)`** — computes the BCE loss to monitor training convergence.
3. **`gradient_descent(X, y, ...)`** — runs the weight-update loop with class weighting and L2 regularization, returning the trained weights and loss history.

Training runs for **3,000 iterations** with `learning_rate=0.5` and L2 penalty `lambda_=0.01`.

In [ ]:
def sigmoid(z):
    """Computes the sigmoid of z."""
    return 1 / (1 + np.exp(-z))

def compute_loss(y_true, y_pred):
    """Computes the Binary Cross-Entropy Loss."""
    epsilon = 1e-15
    loss = -np.mean(y_true * np.log(y_pred + epsilon) + (1 - y_true) * np.log(1 - y_pred + epsilon))
    return loss

def gradient_descent(X, y, learning_rate=0.5, iterations=3000, lambda_=0.01):
    """
    Optimizes weights using gradient descent with:
    - Class weighting to handle imbalanced data
    - L2 regularization to prevent overfitting (bias term excluded)
    - lambda_: L2 penalty strength (higher = stronger regularization)
    """
    n_samples, n_features = X.shape
    weights = np.zeros((n_features, 1))
    loss_history = []

    # Class weights: minority (yes) class gradient amplified by neg/pos ratio
    pos = (y == 1).sum()
    neg = (y == 0).sum()
    weight_vec = np.where(y == 1, neg / pos, 1.0)  # shape (N, 1)

    # L2 penalty mask: do not regularize the bias term (index 0)
    l2_mask = np.ones((n_features, 1))
    l2_mask[0] = 0.0

    for i in range(iterations):
        # 1. Forward Pass
        z = np.dot(X, weights)
        y_hat = sigmoid(z)

        # 2. Weighted Gradient + L2 penalty
        gradient = np.dot(X.T, (y_hat - y) * weight_vec) / n_samples
        gradient += (lambda_ / n_samples) * weights * l2_mask

        # 3. Update Weights
        weights -= learning_rate * gradient

        if i % 100 == 0:
            current_loss = compute_loss(y, y_hat)
            loss_history.append(current_loss)

    return weights, loss_history

# Run the training
weights, loss_history = gradient_descent(X_train, y_train, learning_rate=0.5, iterations=3000, lambda_=0.01)

## Step 3: Generate Predictions and Verify Convergence

We define a `predict()` function that applies the sigmoid to $\mathbf{X}\mathbf{w}$ and thresholds at **0.4** to produce binary labels. A threshold below 0.5 recovers more positive predictions, compensating for class imbalance.

We also plot the BCE loss curve over 3,000 iterations to confirm that gradient descent has converged.

In [ ]:
def predict(X, weights, threshold=0.4):
    """Convert raw probabilities to binary class labels using a threshold."""
    probabilities = sigmoid(np.dot(X, weights))
    return (probabilities >= threshold).astype(int)

# Quick accuracy check
y_pred_train = predict(X_train, weights)
y_pred_test  = predict(X_test,  weights)

train_accuracy = np.mean(y_pred_train == y_train)
test_accuracy  = np.mean(y_pred_test  == y_test)
print(f"Train Accuracy: {train_accuracy * 100:.2f}%")
print(f"Test  Accuracy: {test_accuracy  * 100:.2f}%  (note: misleading under class imbalance)")

# Loss convergence curve
plt.plot(range(0, 3000, 100), loss_history, color="#378ADD")
plt.title("Binary Cross-Entropy Loss — Training Convergence")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.show()

## Step 4: Compute Probabilities at Optimal Threshold

Step 3 used a fixed `threshold=0.4` for a quick accuracy check. Here we explicitly extract the raw sigmoid probabilities and re-apply `threshold=0.4` to produce the predictions used for the full evaluation in Steps 5–6.

Separating the probability computation from the threshold step makes it easy to sweep thresholds in Step 7 without re-running the model.

In [ ]:
# Step 4.1: Predicted Probabilities & Labels
# Get raw probabilities from the sigmoid output, then apply threshold=0.4
# threshold=0.4 chosen to maximise recall (0.765) while keeping
# precision at a reasonable level (~1 in 5 calls converts)
prob_train   = sigmoid(np.dot(X_train, weights))
prob_test    = sigmoid(np.dot(X_test,  weights))
y_pred_train = (prob_train >= 0.4).astype(int)
y_pred_test  = (prob_test  >= 0.4).astype(int)

## Step 5: Evaluation Metrics

We evaluate the model using four standard classification metrics, all computed manually from the **confusion matrix** without any library functions.

### Confusion Matrix

| | Predicted Negative | Predicted Positive |
|---|---|---|
| **Actual Negative** | TN (True Negative) | FP (False Positive) |
| **Actual Positive** | FN (False Negative) | TP (True Positive) |

### Accuracy
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$
> **Caution:** Under class imbalance (~88 % negative), a model that always predicts "no" achieves 88 % accuracy. Accuracy alone is a misleading metric here.

### Precision
$$\text{Precision} = \frac{TP}{TP + FP}$$

### Recall (Sensitivity)
$$\text{Recall} = \frac{TP}{TP + FN}$$

### F1-Score
$$F_1 = \frac{2 \times \text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

The **F1-score** is the primary evaluation metric for this imbalanced dataset, as it rewards models that are both precise and comprehensive in identifying subscribers.

In [ ]:
def confusion_matrix(y_true, y_pred):
    """Return (TP, FP, TN, FN) from true and predicted binary labels."""
    TP = int(((y_pred == 1) & (y_true == 1)).sum())
    FP = int(((y_pred == 1) & (y_true == 0)).sum())
    TN = int(((y_pred == 0) & (y_true == 0)).sum())
    FN = int(((y_pred == 0) & (y_true == 1)).sum())
    return TP, FP, TN, FN

def compute_metrics(y_true, y_pred, label=""):
    """Print and return accuracy, precision, recall, F1, and specificity."""
    TP, FP, TN, FN = confusion_matrix(y_true, y_pred)
    accuracy    = (TP + TN) / (TP + FP + TN + FN)
    precision   = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall      = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1          = (2 * precision * recall / (precision + recall)
                   if (precision + recall) > 0 else 0.0)
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

    print(f"\n{'='*45}\n  {label}\n{'='*45}")
    print(f"  Confusion Matrix:\n            Pred no   Pred yes")
    print(f"  True no    {TN:>6}    {FP:>6}   (TN / FP)")
    print(f"  True yes   {FN:>6}    {TP:>6}   (FN / TP)")
    print(f"\n  Accuracy   : {accuracy:.4f}  — unreliable under class imbalance")
    print(f"  Precision  : {precision:.4f}  — of predicted yes, how many are truly yes")
    print(f"  Recall     : {recall:.4f}  — of true yes, how many did we catch")
    print(f"  F1 Score   : {f1:.4f}  — harmonic mean of precision & recall")
    print(f"  Specificity: {specificity:.4f}  — of true no, how many did we correctly reject")
    return {"accuracy": accuracy, "precision": precision, "recall": recall,
            "f1": f1, "TP": TP, "FP": FP, "TN": TN, "FN": FN}

train_metrics = compute_metrics(y_train, y_pred_train, "Train Set")
test_metrics  = compute_metrics(y_test,  y_pred_test,  "Test Set")

## Step 6: AUC-ROC and Precision-Recall Curves

Two threshold-independent metrics provide a complete picture of model performance across all operating points:

### AUC-ROC
The area under the Receiver Operating Characteristic curve measures the model's ability to separate classes regardless of threshold:
- **1.0** = perfect classifier
- **0.5** = random classifier (diagonal baseline)
- **> 0.75** is generally considered useful in practice

### AUC-PR (Precision-Recall Curve)
More informative than AUC-ROC under class imbalance. The no-skill baseline equals the positive class rate (~0.117), so any AUC-PR significantly above this indicates meaningful signal.

Both curves are computed manually via the trapezoidal rule across 200 threshold values, with no sklearn dependencies.

In [ ]:
# Step 4.3: AUC-ROC & Precision-Recall Curves (manual, no sklearn)
def roc_auc(y_true, prob):
    """Compute ROC curve points and AUC via trapezoidal rule."""
    thresholds = np.linspace(0, 1, 200)
    tprs, fprs = [], []
    y_flat = y_true.flatten()
    for t in thresholds:
        pred = (prob.flatten() >= t).astype(int)
        TP = int(((pred == 1) & (y_flat == 1)).sum())
        FP = int(((pred == 1) & (y_flat == 0)).sum())
        TN = int(((pred == 0) & (y_flat == 0)).sum())
        FN = int(((pred == 0) & (y_flat == 1)).sum())
        tprs.append(TP / (TP + FN) if (TP + FN) > 0 else 0)
        fprs.append(FP / (FP + TN) if (FP + TN) > 0 else 0)
    fprs_arr   = np.array(fprs);  tprs_arr = np.array(tprs)
    sorted_idx = np.argsort(fprs_arr)
    fprs_s     = fprs_arr[sorted_idx];  tprs_s = tprs_arr[sorted_idx]
    auc        = float(np.trapz(tprs_s, fprs_s))
    return fprs_s, tprs_s, auc

def pr_curve(y_true, prob):
    """Compute precision-recall curve points across thresholds."""
    thresholds = np.linspace(0, 1, 200)
    precisions, recalls = [], []
    y_flat = y_true.flatten()
    for t in thresholds:
        pred = (prob.flatten() >= t).astype(int)
        TP = int(((pred == 1) & (y_flat == 1)).sum())
        FP = int(((pred == 1) & (y_flat == 0)).sum())
        FN = int(((pred == 0) & (y_flat == 1)).sum())
        precisions.append(TP / (TP + FP) if (TP + FP) > 0 else 1.0)
        recalls.append(   TP / (TP + FN) if (TP + FN) > 0 else 0.0)
    return np.array(recalls), np.array(precisions)

fprs_tr, tprs_tr, auc_tr = roc_auc(y_train, prob_train)
fprs_te, tprs_te, auc_te = roc_auc(y_test,  prob_test)
rec_tr,  pre_tr           = pr_curve(y_train, prob_train)
rec_te,  pre_te           = pr_curve(y_test,  prob_test)

print(f"  AUC-ROC  Train : {auc_tr:.4f}")
print(f"  AUC-ROC  Test  : {auc_te:.4f}")
print(f"  (0.5 = random guess, 1.0 = perfect, >0.75 is generally usable)")

In [ ]:
# Step 4.4: Visualisation 
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1 — Confusion Matrix heatmap (test set)
cm_vals = np.array([[test_metrics["TN"], test_metrics["FP"]],
                    [test_metrics["FN"], test_metrics["TP"]]])
ax = axes[0]
im = ax.imshow(cm_vals, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Pred no", "Pred yes"])
ax.set_yticklabels(["True no", "True yes"])
ax.set_title("Confusion Matrix (Test Set)")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_vals[i, j]), ha="center", va="center",
                color="white" if cm_vals[i, j] > cm_vals.max() / 2 else "black",
                fontsize=14, fontweight="bold")
plt.colorbar(im, ax=ax)

# Plot 2 — ROC Curve
ax = axes[1]
ax.plot(fprs_tr, tprs_tr, color="#378ADD", lw=1.5, label=f"Train AUC = {auc_tr:.3f}")
ax.plot(fprs_te, tprs_te, color="#534AB7", lw=2,   label=f"Test  AUC = {auc_te:.3f}")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4, label="Random baseline")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve"); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Plot 3 — Precision-Recall Curve (more informative under class imbalance)
ax = axes[2]
ax.plot(rec_tr, pre_tr, color="#378ADD", lw=1.5, label="Train")
ax.plot(rec_te, pre_te, color="#534AB7", lw=2,   label="Test")
baseline = float(y_test.mean())
ax.axhline(baseline, color="gray", lw=1, linestyle="--",
           label=f"No-skill baseline ({baseline:.2f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve"); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("evaluation_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 7: Threshold Sensitivity Analysis

The default threshold of 0.4 was chosen to increase recall relative to 0.5, but it is not necessarily optimal. We sweep thresholds from 0.3 to 0.7 and report Precision, Recall, and F1 at each point on the test set:

- **Lower threshold** → higher Recall (catch more subscribers), lower Precision (more wasted calls)
- **Higher threshold** → higher Precision (fewer wasted calls), lower Recall (miss more subscribers)

The threshold that maximises F1 gives the best balanced operating point for this dataset.

In [ ]:
# Step 4.5: Threshold Sensitivity Analysis 
# Different thresholds trade off precision vs recall.
# Bank marketing prioritises recall (missing a subscriber is costly),
# but threshold=0.6 maximises F1 for this dataset.
print(" Threshold Sensitivity (Test Set)")
print(f"  {'Threshold':>10}  {'Precision':>10}  {'Recall':>8}  {'F1':>8}")
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (prob_test.flatten() >= t).astype(int)
    y_f  = y_test.flatten()
    TP = int(((pred==1)&(y_f==1)).sum()); FP = int(((pred==1)&(y_f==0)).sum())
    FN = int(((pred==0)&(y_f==1)).sum())
    p  = TP / (TP + FP) if (TP + FP) > 0 else 0
    r  = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    print(f"  {t:>10.1f}  {p:>10.4f}  {r:>8.4f}  {f1:>8.4f}")
print("\n  Recommendation: threshold=0.6 maximises F1 for this dataset")

## Business Interpretation

By targeting only the customers flagged as likely subscribers by the model (at the F1-maximizing threshold), the bank can reach a large fraction of potential subscribers while contacting far fewer people than a random calling strategy would require.

For example, at the optimal threshold the model can **reach ~77 % of all potential subscribers by contacting only ~48 % of the customer base**, a substantial efficiency gain over unguided outreach. The exact numbers depend on the chosen threshold: a lower threshold catches more subscribers at the cost of more wasted calls; a higher threshold is more conservative but misses more opportunities.

## Summary of Results

### Model Configuration

| Parameter | Value |
|-----------|-------|
| Learning rate ($\alpha$) | 0.5 |
| Iterations | 3,000 |
| L2 penalty ($\lambda$) | 0.01 |
| Class weighting | Automatic ($n_{\text{neg}} / n_{\text{pos}} \approx 7.5\times$) |
| Decision threshold | 0.4 (default); see Step 7 for sensitivity |
| Data split | 80 % train / 20 % test (stratified) |

### Performance

See the Step 5 code output for Precision, Recall, and F1 at threshold = 0.4, and Step 6 for AUC-ROC and AUC-PR values.

### Key Observations

1. **BCE loss is convex.** Gradient descent is guaranteed to converge to the global minimum, unlike non-convex losses in neural networks. The convergence curve in Step 3 confirms this.

2. **Accuracy is misleading.** The dataset is ~88 % negative — a model that always predicts "no" achieves 88 % accuracy. F1-score is the primary metric.

3. **Class weighting is essential.** Without upweighting minority-class gradients, the model converges to predicting "no" for nearly everyone and achieves ~88 % accuracy by doing nothing useful.

4. **Weights are directly interpretable.** Unlike the Decision Tree (which produces a non-linear rule set), Logistic Regression produces a single set of weights that directly quantify each feature's log-odds contribution to the subscription probability.

5. **Decision Tree vs. Logistic Regression.** Logistic Regression assumes a single linear decision boundary in the feature space. The Decision Tree partitions the feature space into rectangular regions, capturing non-linear interactions without feature engineering — but at the cost of the elegant interpretability of individual weights.

## Step 8: Feature Importance

In Logistic Regression, the learned weights **are** the feature importances. The absolute value of each weight indicates how strongly that feature influences the predicted probability; the sign indicates the direction:

- **Positive weight** → feature increases $P(\text{subscribe})$
- **Negative weight** → feature decreases $P(\text{subscribe})$

This is a direct, interpretable measure, unlike tree-based importance scores which aggregate Gini gain across many splits. The bias term (index 0) is excluded from the ranking.

In [ ]:
#  Step 5: Feature Importance 
# In logistic regression, the learned weights ARE the feature importances.
# Larger absolute value = stronger influence on the predicted probability.
# Positive weight → feature increases P(yes); negative → decreases P(yes).

feature_names = list(X_raw.columns)
importances   = weights[1:].flatten()   # index 0 is the bias term — excluded

sorted_idx = np.argsort(np.abs(importances))[::-1]
top_n = 15

print("Top 15 Most Important Features:")
print(f"  {'Feature':<35} {'Weight':>8}  Direction")
print("  " + "─" * 56)
for i in sorted_idx[:top_n]:
    direction = "→ more likely YES" if importances[i] > 0 else "→ more likely NO"
    print(f"  {feature_names[i]:<35} {importances[i]:>+8.4f}  {direction}")

# Horizontal bar chart — blue = positive effect, red = negative effect
plt.figure(figsize=(10, 6))
top_names   = [feature_names[i] for i in sorted_idx[:top_n]]
top_weights = [importances[i]   for i in sorted_idx[:top_n]]
colors = ["#378ADD" if w > 0 else "#E05C5C" for w in top_weights]

plt.barh(top_names[::-1], top_weights[::-1], color=colors[::-1])
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Top 15 Feature Importances (Logistic Regression Weights)")
plt.xlabel("Weight  (positive = increases P(subscribe), negative = decreases)")
plt.tight_layout()
plt.show()